# Phase 7 — TOR3-MALAM raw / DEV-shortlisted temporal comparison

Actual video inference in Kaggle, GPU0 detector and external encoder, CPU BoT-SORT association. This notebook consumes a frozen DEV-only shortlist and user-approved N1 geometry; it cannot choose an enhancement from holdout outcomes. Independent fullvideo variants preserve continuous source history. Outputs are machine decisions pending physical-passage review, not an accuracy benchmark.

The same raw/enhanced frame feeds detector and encoder. Source-bound Phase6 B v1 is loaded with exactly one measurement-provenance literal adaptation to Phase7; no earlier source/evidence is edited. Numerical controls remain the PAGI experimental controls; they are not night calibration or production defaults.


In [ ]:
from pathlib import Path
import json, sys
REPO_ROOT = Path(globals().get('REPO_ROOT', Path.cwd())).resolve()
CONFIG_PATH = REPO_ROOT / 'docs/evidence/phase_7/config/TOR3-MALAM_temporal_v1.json'
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Attach the repository/source bundle with frozen DEV shortlist and temporal configuration: ' + str(CONFIG_PATH))
sys.path.insert(0, str(REPO_ROOT / 'notebooks'))
CONFIG = json.loads(CONFIG_PATH.read_text())
CONFIG['repo_root'] = str(REPO_ROOT)
# VIDEO_PATH and all variant/settings selections are explicit in this bound config.
VIDEO_PATH = Path(CONFIG['video_path'])
print('Video:', VIDEO_PATH)
print('Variants:', CONFIG['variants'])


In [ ]:
# Validate actual input bytes, approved geometry and DEV-only selection before models.
from phase_7_temporal_helpers import verify_temporal_inputs
BOUND_INPUTS = verify_temporal_inputs(CONFIG)
print('Verified DEV shortlist:', BOUND_INPUTS['selection'])
print('Geometry:', BOUND_INPUTS['geometry_path'])


## Runtime
Pinned packages reproduce the previously verified detector/tracker/encoder integration. GPU0 is mandatory for detector and encoder; no CPU or substitute-model fallback. CUDA graph partitions assigned to CPU are reported separately. Changing packages in an already imported session requires a restart.


In [ ]:
import importlib.metadata as metadata
import subprocess
WANTED = {'ultralytics':'8.4.175', 'lap':'0.5.13', 'onnx':'1.20.1', 'onnxruntime-gpu':'1.24.4'}
missing = []
for package, version in WANTED.items():
    try: installed = metadata.version(package)
    except metadata.PackageNotFoundError: installed = None
    if installed != version:
        imported = 'onnxruntime' if package == 'onnxruntime-gpu' else package.replace('-', '_')
        if imported in sys.modules:
            raise RuntimeError(package + ' already imported with another version; restart the session')
        missing.append(package + '==' + version)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', *missing], check=True)
for package, version in WANTED.items():
    assert metadata.version(package) == version
from phase_7_night_helpers import verify_gpu_device
print(verify_gpu_device(0))


In [ ]:
# Independent software fixtures are required before actual inference.
from phase_7_temporal_tests import run_tests
GUARD_REPORT = run_tests()
GUARD_DIR = Path(CONFIG['output_root']) / 'pre_execution_guard'
GUARD_DIR.mkdir(parents=True, exist_ok=True)
(GUARD_DIR / 'synthetic_validation.json').write_text(json.dumps(GUARD_REPORT, indent=2) + '\n')
if not GUARD_REPORT['successful']:
    raise RuntimeError('Independent Phase7 temporal fixtures failed; actual model execution blocked')
print('Independent synthetic tests:', GUARD_REPORT['tests_passed'], '/', GUARD_REPORT['tests_run'])


## Actual independent fullvideo sequences
Two passes start at source frame0 with a fresh tracker per variant; there is no reset at development/holdout window boundaries. Detector, tracker, ReID, geometry and counting settings are fixed. Counting comparison is limited to the declared five DEV/holdout windows; track IDs are dynamically discovered there, with complete source histories and raw competitor context retained. This is not a whole-video traffic subtotal. Full vehicle-episode classification remains offline. Do not match physical passages using equal track IDs between variants. Review any passage crossing a split boundary as leakage/independence-unknown before using it for holdout conclusions.


In [ ]:
from phase_7_temporal_helpers import run_night_temporal
EXECUTION = run_night_temporal(CONFIG)
RUN_DIR = Path(EXECUTION['run_dir'])
# Preserve the guard with actual run artifacts; its fixtures are not video findings.
import shutil
shutil.copytree(GUARD_DIR, RUN_DIR / 'pre_execution_guard', dirs_exist_ok=False)
# Extend the new run manifest with the actual pre-execution guard copied above.
from phase_7_night_helpers import sha256_file
MANIFEST_PATH = RUN_DIR / 'retention_manifest.json'
MANIFEST = json.loads(MANIFEST_PATH.read_text())
MANIFEST['artifacts'] = [{'path': str(path.relative_to(RUN_DIR)), 'sha256': sha256_file(path), 'bytes': path.stat().st_size}
                         for path in sorted(RUN_DIR.rglob('*')) if path.is_file() and path.name != 'retention_manifest.json']
MANIFEST_PATH.write_text(json.dumps(MANIFEST, indent=2, sort_keys=True) + '\n')
print('Retained actual output:', RUN_DIR)


In [ ]:
REPORT = EXECUTION['report']
print('Status:', REPORT['status'])
for name, variant in REPORT['variants'].items():
    print(name, 'frames:', variant['frames_processed'])
    print('Machine counting candidates:', variant['counting_report']['decision_status_counts'])
    print('Resource:', variant['resource'])
print('Physical-passage review remains:', REPORT['manual_review_status'])


## Required next review
Retain decoded raw/input examples, source-linked detections/tracks, real CUDA profile evidence, timing/resource and the frozen source bundle. On the same physical passage, compare missed/extra observations, fragmentation, wrong merges, duplicates, category conflicts and measured vehicle BC crossing. Keep machine results separate from any explicit manual overlay. Fewer pending events, higher confidence and brighter pixels alone do not demonstrate improvement. No ground truth exists; report bounded actual observations and regressions without accuracy/precision/recall/F1/MAPE. Do not start Phase8 from this notebook.
